# NB2 · 值迭代与策略迭代：亲手兑现压缩映射的承诺

第四讲（L4）把「解 Bellman 最优方程」落成两个可执行的算法。这本笔记本在 4×4 作业世界上亲手实现它们，并用实验逐条审计第四讲的两条承重结论：

1. **实现**值迭代与策略迭代，验证两条路收敛到**同一个 v\***（差 < 1e-6）；
2. 画出两法**收敛曲线**，看「几何式收敛」与「有限步终止」的不同长相；
3. **γ 扫描** γ ∈ {0.5, 0.9, 0.99}，验证「γ 越接近 1，压得越慢」；
4. 用**值迭代误差界** γᵏ/(1−γ)·‖v₀−v₁‖ 预测每轮误差，与实测逐轮对账。

**前置**：NB0（环境复刻与张量化——本本自包含，不依赖它也能跑）+ 第四讲正文，尤其站内 L4 的两条推导链——**Banach 压缩映射 → Bellman 最优算子是 γ-压缩 → 不动点存在唯一**，与**值迭代误差界**。这两条定理今天不再是「看过的结论」，而是被你的代码逐轮审计的对象。

预估 45 分钟 · 难度 2/3 · 三处 **TODO** 由你完成，全部 **✅ 自检格**跑绿即毕业。

## 怎么用这本笔记本

- **TODO 格**：给全函数签名与形状提示，不给实现——写完后删掉格里的 `raise NotImplementedError(...)` 再往下跑；
- **✅ 自检格**：assert + 绿字结论，跑绿才算完成；哪格红了，问题多半在它上面最近的 TODO 里；
- **🏔 挑战格**：开放任务，无标准答案，不影响跑绿；
- 选中格子按 **Shift + Enter** 运行；**刷新浏览器会清空 kernel**，回来 Run All 重跑一遍即可；
- 带宽：本本要画图，浏览器首次需再下载 ~8–10MB 的 matplotlib（之后强缓存）。

## 0 · 环境自检

照例先看清脚下：Python 在哪跑、numpy 与 matplotlib 是否就绪。

In [ ]:
import sys
import platform

try:
    import pyodide  # noqa: F401  只有浏览器内核（Pyodide）里能导入
    IN_BROWSER = True
except ImportError:
    IN_BROWSER = False

assert sys.implementation.name == "cpython"
print("Python :", sys.version.split()[0], "|", platform.platform())
print("运行环境 :", "浏览器（Pyodide / WebAssembly）" if IN_BROWSER else "本地 Python（nbconvert 校验模式）")

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("numpy :", np.__version__, "  matplotlib :", matplotlib.__version__)
assert int(np.__version__.split(".")[0]) in (1, 2)
print("numpy + matplotlib 就绪——本本要画收敛曲线")

## 1 · 前置理论：两条定理，今天逐轮审计

**推导一 · 压缩映射 → 唯一不动点。** Bellman 最优算子

$$(T^*v)(s) = \max_a \Big[\, R[s,a] + \gamma \sum_{s'} T[s,a,s']\, v(s') \Big]$$

在 ∞-范数下是系数恰为 γ 的**压缩映射**；Banach 不动点定理随即给出三连：**不动点存在、唯一、从任意初值出发迭代都收敛**，且白送一条速率——**误差每轮精确乘一次 γ**。
→ 审计方式：实验 1（两法同 v\*）、实验 3（γ 扫描）。

**推导二 · 值迭代的误差界。** 从 $v_0$ 出发迭代 k 轮后，

$$\|v_k - v^*\|_\infty \;\le\; \frac{\gamma^k}{1-\gamma}\,\|v_0 - v_1\|_\infty$$

它把「不可观测的真实误差」换成「可观测的相邻两轮之差」。
→ 审计方式：实验 4（预测 vs 实测，逐轮对账）。

记号说死：本本里 $v_1 = T^*v_0$，所以 $\|v_0 - v_1\|_\infty = \|v_0 - T^*v_0\|_\infty$——同一个量的两种写法。

## 2 · 4×4 GridWorld：自包含复刻

与 NB0 / NB1 完全一致的作业世界（每本笔记本自包含，不依赖其他本的变量），规格即主站 L1「代码精讲」的 A4 配置：

| 项目 | 值 |
|---|---|
| 网格 | **4×4**，状态编号 s1–s16 |
| 起点 | **s1**（左上角） |
| 禁区 | **s8、s10** |
| 目标 | **s12** |
| 奖励 | 边界 **−1** / 禁区 **−1** / 目标 **+1** / 其他 **0** |
| 折扣 | **γ = 0.9** |

编号规则 **s_i ↔ ((i−1) % 4, (i−1) // 4)**，y 向下增长。动作 5 个，列序**下、右、上、左、原**。分支优先级**出界 > 目标 > 禁区 > 普通**；禁区是**弹回**的（原地不动、挨罚 −1）。

In [ ]:
SIZE = 4
NUM_STATES = SIZE * SIZE          # s1 .. s16
START, TARGET = 1, 12
FORBIDDEN = {8, 10}
REWARDS = {"boundary": -1.0, "forbidden": -1.0, "target": 1.0, "other": 0.0}
GAMMA = 0.9                       # 与主站 A4 作业世界一致

assert (SIZE, NUM_STATES, START, TARGET) == (4, 16, 1, 12)
assert FORBIDDEN == {8, 10} and REWARDS["target"] == 1.0
print("规格就绪：4×4 / 起点 s1 / 禁区 s8,s10 / 目标 s12 / γ =", GAMMA)

In [ ]:
ACTION_SPACE = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 0)]  # 下 右 上 左 原
ARROWS = ["↓", "→", "↑", "←", "•"]                          # 同一下标序，打印策略用

def s2xy(s):
    # 状态编号 → (x, y)，y 向下增长：s1=(0,0) s8=(3,1) s10=(1,2) s12=(3,2)
    i = int(s) - 1
    return i % SIZE, i // SIZE

def xy2s(x, y):
    return int(y) * SIZE + int(x) + 1

assert s2xy(12) == (3, 2) and all(xy2s(*s2xy(s)) == s for s in range(1, 17))
print("动作空间（作业代码列序）:", ACTION_SPACE)

In [ ]:
class GridWorld:
    """4×4 作业世界：NB0 瘦身复刻（自包含单文件版）。

    语义（作业代码规则，分支优先级 出界 > 目标 > 禁区 > 普通）：
      出界 → 原地，−1；进目标 → 走进，+1；撞禁区 → 弹回原地，−1；普通/原 → 移动/不动，0
    """

    def __init__(self, size=SIZE, start=START, target=TARGET, forbidden=FORBIDDEN):
        self.size = size
        self.num_states = size * size
        self.start_state, self.target_state = start, target
        self.forbidden_states = set(forbidden)
        self.action_space = ACTION_SPACE
        self.agent_state = start

    def reset(self):
        self.agent_state = self.start_state
        return self.agent_state

    def _get_next_state_and_reward(self, state, action):
        x, y = s2xy(state)
        nxt = np.array([x, y]) + np.array(action)
        if not (0 <= nxt[0] < self.size and 0 <= nxt[1] < self.size):
            next_state, reward = state, REWARDS["boundary"]        # 1) 出界
        elif xy2s(nxt[0], nxt[1]) == self.target_state:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["target"]   # 2) 目标
        elif xy2s(nxt[0], nxt[1]) in self.forbidden_states:
            next_state, reward = state, REWARDS["forbidden"]       # 3) 禁区：弹回
        else:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["other"]    # 4) 普通
        return next_state, reward

    def step(self, action):
        next_state, reward = self._get_next_state_and_reward(self.agent_state, action)
        done = next_state == self.target_state
        self.agent_state = next_state
        return next_state, reward, done, {}

env = GridWorld()
assert env._get_next_state_and_reward(1, (0, -1)) == (1, -1.0)     # s1 上：出界
assert env._get_next_state_and_reward(11, (1, 0)) == (12, 1.0)     # s11 右：进目标
assert env._get_next_state_and_reward(7, (1, 0)) == (7, -1.0)      # s7 右：撞禁区弹回
print("环境就绪：三条锚点转移与站内作业世界一致")

In [ ]:
# 张量化：确定性转移 one-hot 张量 T[s,a,s'] 与奖励矩阵 R[s,a]
T = np.zeros((NUM_STATES, 5, NUM_STATES))
R = np.zeros((NUM_STATES, 5))
for s in range(1, NUM_STATES + 1):
    for a in range(5):
        ns, r = env._get_next_state_and_reward(s, ACTION_SPACE[a])
        T[s - 1, a, ns - 1] = 1.0
        R[s - 1, a] = r

assert T.shape == (16, 5, 16) and R.shape == (16, 5)
assert np.allclose(T.sum(axis=-1), 1.0)          # one-hot：每行恰一个 1
assert T[11, 4, 11] == 1.0 and R[11, 4] == 1.0   # s12 原地领奖：+1 自环
print("T:", T.shape, " R:", R.shape, "——动力学张量化完成，后面全是矩阵运算")

## 3 · 值迭代：每轮一次「贪心自举」——TODO 1

第四讲 §4.1 的单轮节奏：**策略更新**挑动作、**价值更新**走一步，逐元素合并成一行——

$$v_{k+1}(s) = \max_a \Big[\, R[s,a] + \gamma \sum_{s'} T[s,a,s']\, v_k(s') \Big]$$

实现提示：**不要写状态双重循环**。16 状态 × 5 动作的整张 q 表一次算完——`q = R + gamma * np.einsum("sat,t->sa", T, v)` 得 (16, 5)；`q.max(axis=1)` 是价值更新，`q.argmax(axis=1)` 是策略更新。停机条件：相邻两轮之差 ‖v_{k+1} − v_k‖∞ < tol——注意这**只是截断准则**，不是真实误差（§8 回来算这笔账）。

In [ ]:
# ══════════════════ TODO 1 · 值迭代 ══════════════════
def value_iteration(T, R, gamma, tol=1e-10, max_iter=20000):
    """值迭代：v_{k+1} = T*v_k，直到相邻两轮之差小于 tol。

    参数
    ----
    T     : (16, 5, 16) 转移张量 T[s, a, s']
    R     : (16, 5)     奖励矩阵 R[s, a]
    gamma : 折扣因子
    tol   : 停机阈值（相邻两轮的 ‖·‖∞）
    max_iter : 安全上限

    返回
    ----
    v_star    : (16,)        最优状态价值
    pi_star   : (16,) int    最优策略（动作下标 0=下 1=右 2=上 3=左 4=原）
    diff_hist : (K,)         每轮的 ‖v_k − T*v_k‖∞（K = 迭代轮数）
    """
    v = np.zeros(R.shape[0])
    diff_hist = []
    # 提示：
    #   1) q = R + gamma * np.einsum("sat,t->sa", T, v)    # 整张 q 表，shape (16, 5)
    #   2) v_new = q.max(axis=1)；记录 diff 后判断是否 < tol
    #   3) 收敛后 pi_star = q.argmax(axis=1)
    # TODO: 实现值迭代主体（写完删除下面这行）
    raise NotImplementedError("TODO 1：完成值迭代后删除本行")

## 4 · 策略迭代：评估与改进的华尔兹——TODO 2

每轮两步：

- **PE 策略评估**：解 Bellman 期望方程 $v_\pi = r_\pi + \gamma P_\pi v_\pi$，移项即线性方程组 $(I - \gamma P_\pi)\, v_\pi = r_\pi$——本本用 `np.linalg.solve` 求**精确解**（NB1 已验证它与迭代解殊途同归）；
- **PI 策略改进**：$q = R + \gamma\,(T v_\pi)$，逐状态换上 q 最大的动作（`argmax(axis=1)`）。

取出策略 π 的转移矩阵与奖励向量各一行 fancy indexing：`P_pi = T[np.arange(nS), pi]` 得 (16, 16)、`r_pi = R[np.arange(nS), pi]` 得 (16,)。

**终止**：新策略与旧策略完全相同即停——引理 4.1（每轮严格改进或已最优）保证同一策略不会第二次出现，至多 $|A|^{|S|} = 5^{16}$ 轮内**精确终止**（实际轮数远小于它）。

In [ ]:
# ══════════════════ TODO 2 · 策略迭代 ══════════════════
def policy_iteration(T, R, gamma, max_iter=100):
    """策略迭代：外层「评估 → 改进」，策略不再变化即精确终止。

    返回
    ----
    v_star  : (16,)      策略稳定那轮 PE 解出的价值（即 v*）
    pi_star : (16,) int  最优策略
    v_hist  : (K, 16)    每轮 PE 解出的 v_π 依次堆叠（§6 画收敛曲线用）
    """
    nS = R.shape[0]
    pi = np.zeros(nS, dtype=int)      # 初始策略：全选动作 0（下）——任意初值都行
    v_hist = []
    # 提示（一轮的骨架）：
    #   PE: P_pi = T[np.arange(nS), pi];  r_pi = R[np.arange(nS), pi]
    #       v = np.linalg.solve(np.eye(nS) - gamma * P_pi, r_pi)
    #   v_hist.append(v.copy())
    #   PI: pi_new = (R + gamma * np.einsum("sat,t->sa", T, v)).argmax(axis=1)
    #   pi_new 与 pi 完全相同 → break（返回此轮的 v 与 pi）
    # TODO: 实现策略迭代主体（写完删除下面这行）
    raise NotImplementedError("TODO 2：完成策略迭代后删除本行")

## 5 · 对账：两条路应殊途同归

第四讲定理 4.1 的证明把两法从同一起点并排跑，归纳可证 $v_k \le v_{\pi_k} \le v^*$：**策略迭代每步都压着值迭代打**，而值迭代已知收敛到 v\*。既然殊途同归，两法解出的 v\* 之差应在 1e-6 以内——下面的自检格兑现这句话。

In [ ]:
# ✅ 自检 1：两法解出的 v* 应逐格一致（差 < 1e-6）
v_vi, pi_vi, hist_vi = value_iteration(T, R, GAMMA)
v_pi, pi_pi, v_hist_pi = policy_iteration(T, R, GAMMA)

gap = float(np.abs(v_vi - v_pi).max())
print(f"值迭代  ：{len(hist_vi)} 轮收敛（默认 tol = 1e-10）")
print(f"策略迭代：{len(v_hist_pi)} 轮外层迭代，每轮含一次 16×16 线性求解")
print(f"两法 v* 的最大差 ‖v*_VI − v*_PI‖∞ = {gap:.2e}")

assert gap < 1e-6, "两条路应收敛到同一个 v*——差应在 1e-6 以内"
print("✅ 自检通过：值迭代与策略迭代殊途同归（v* 差 < 1e-6）")
print("   最优策略也", "完全一致" if (pi_vi == pi_pi).all() else "存在并列 argmax 的差异（价值相等即都对）")

In [ ]:
def show_policy(pi, title=""):
    # 把 (16,) 策略下标打印成 4×4 箭头网格（第 1 行是 y=0）
    if title:
        print(title)
    for row in pi.reshape(SIZE, SIZE):
        print("   ", " ".join(ARROWS[int(a)] for a in row))

print("v*（保留 3 位小数）：")
print(np.round(v_vi.reshape(SIZE, SIZE), 3))
show_policy(pi_vi, "\n最优策略（• = 原地领奖）：")

assert abs(v_vi[11] - 1 / (1 - GAMMA)) < 1e-4, "v*(s12) 应等于原地领奖的几何级数 1/(1−γ)"
print(f"\nv*(s12) = {v_vi[11]:.4f} = 1/(1−γ)：目标上的最优动作是「原地领奖」，价值是每步 +1 的几何级数")

## 6 · 收敛曲线：几何式 vs 有限步

- 值迭代：横轴**迭代轮数**，纵轴每轮的 $\|v_k - T^*v_k\|_\infty$（就是 TODO 1 返回的 `diff_hist`）；
- 策略迭代：横轴**外层轮数**，纵轴取相邻两次 PE 解出的价值之差 $\|v_{\pi_{k+1}} - v_{\pi_k}\|_\infty$。

口径相同（各算一轮的价值变化量），可同图比较。log 纵轴下会看到两种完全不同的形状。

In [ ]:
pi_diffs = np.abs(np.diff(v_hist_pi, axis=0)).max(axis=1)   # 策略迭代外层的相邻价值差

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.semilogy(np.arange(1, len(hist_vi) + 1), hist_vi, lw=2,
            label=f"Value iteration: {len(hist_vi)} sweeps")
ax.semilogy(np.arange(1, len(pi_diffs) + 1), pi_diffs, "o-", lw=2, ms=7,
            label=f"Policy iteration: {len(v_hist_pi)} outer rounds")
ax.set_xlabel("iteration $k$")
ax.set_ylabel(r"$\|v_{k+1}-v_k\|_\infty$ (log scale)")
ax.set_title(f"Convergence on the 4x4 GridWorld ($\\gamma$ = {GAMMA})")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

slope = float(np.polyfit(np.arange(len(hist_vi)), np.log10(hist_vi), 1)[0])
print(f"值迭代曲线的 log₁₀ 斜率 ≈ {slope:.4f}，理论值 log₁₀(γ) = {np.log10(GAMMA):.4f}")
print("—— 每轮误差精确乘一次 γ，压缩映射的速率预言兑现")

读图三个看点：

1. **值迭代的曲线在 log 轴上是一条直线**——几何式收敛的标志，斜率 ≈ log₁₀γ；
2. **策略迭代只有寥寥几个点**，却步步大幅下探——每轮 PE 把当前策略的分数**一次性算到底**（闭式解），信息量远超值迭代的一步自举；定理 4.1 的 $v_k \le v_{\pi_k} \le v^*$ 说它每步都压着值迭代打；
3. **策略迭代最后一步之后策略不再变化**（有限步终止），值迭代理论上永远差一点（渐近收敛）——这也是两法只能「对账到 1e-6」而不能逐位相等的原因。

## 7 · γ 扫描：收敛速度完全被 γ 一手决定——TODO 3

压缩映射白送的速率信息是「每轮误差 ×γ」，于是预测很明确：**γ 越接近 1，达到同一容差所需的轮数越多**。对 γ ∈ {0.5, 0.9, 0.99} 各跑一遍值迭代（统一 tol = 1e-8），记录**迭代轮数**与**最终策略**。

顺带一个值得先押注再揭晓的问题：三个 γ 的最优策略会不同吗？（提示：γ 改变的是价值的**尺度**——v\*(s12) = 1/(1−γ) 会从 2 涨到 100——但动作的**名次**未必跟着变。）

In [ ]:
# ══════════════════ TODO 3 · γ 扫描 ══════════════════
def sweep_gamma(T, R, gammas=(0.5, 0.9, 0.99), tol=1e-8):
    """对每个 γ 跑值迭代，记录迭代轮数与最终策略。

    返回
    ----
    results : dict，键为 γ，值为
        {"iters": int 迭代轮数, "v_star": (16,), "pi_star": (16,) int}
    """
    # 提示：直接复用 TODO 1 的 value_iteration；
    #       len(diff_hist) 就是达到 tol 所用的迭代轮数。
    # TODO: 实现 γ 扫描（写完删除下面这行）
    raise NotImplementedError("TODO 3：完成 γ 扫描后删除本行")

In [ ]:
# ✅ 自检 2：γ 越大，迭代轮数越多
GAMMAS = (0.5, 0.9, 0.99)
results = sweep_gamma(T, R, gammas=GAMMAS)

iters = [results[g]["iters"] for g in GAMMAS]
print("γ      迭代轮数      v*(s1)      v*(s12)")
for g in GAMMAS:
    r = results[g]
    print(f"{g:<6} {r['iters']:>8} {r['v_star'][0]:>10.3f} {r['v_star'][11]:>10.3f}")

assert iters[0] <= iters[1] <= iters[2], "γ 越大压缩越慢：迭代轮数应非降"
assert iters[2] > iters[0], "γ = 0.99 应显著慢于 γ = 0.5"
print()
print("✅ 自检通过：γ 越大迭代轮数越多——收敛速度完全被 γ 一手决定")

In [ ]:
policies_same = all((results[g]["pi_star"] == results[0.5]["pi_star"]).all() for g in GAMMAS)
for g in GAMMAS:
    r = results[g]
    show_policy(r["pi_star"], f"γ = {g} 的最优策略（v*(s12) = {r['v_star'][11]:.4f}）")
    assert abs(r["v_star"][11] - 1 / (1 - g)) < 1e-4 / (1 - g), "v*(s12) 应等于 1/(1−γ)"

print("三个 γ 的最优策略完全相同" if policies_same else "三个 γ 的最优策略存在差异")
print("价值尺度随 γ 涨了 50 倍（2 → 10 → 100），动作名次却纹丝不动——γ 定尺度，名次定策略")

## 8 · 停机准则：把「ε」换算成「真实误差」

实践里值迭代以 $\|v_{k+1} - v_k\|_\infty < \varepsilon$ 停机，但**相邻两轮之差不是当前误差**。站内 L4 推导链的结论——

$$\|v_k - v^*\|_\infty \;\le\; \frac{\gamma^k}{1-\gamma}\,\|v_0 - v_1\|_\infty$$

本世界还有个漂亮的主角：s12 的最优动作是**原地领奖**（+1 自环），于是 $v_k(s12) = 10\,(1 - 0.9^k)$——与书上 3×3 世界 $v_k(s9)$ 的规律一模一样。这条上界在本世界因此**不松**：前一百多轮几乎逐位贴着实测误差。

实验设计：重放一遍值迭代，把每个 $v_k$ 都留下来；v\* 不用迭代近似，而是对最优策略做一次闭式求解（机器精度）；逐轮对账「预测上界 vs 实测误差」。

In [ ]:
# v* 用最优策略下的闭式解（机器精度），不掺迭代的截断误差
P_star = T[np.arange(NUM_STATES), pi_vi]
r_star = R[np.arange(NUM_STATES), pi_vi]
v_star_exact = np.linalg.solve(np.eye(NUM_STATES) - GAMMA * P_star, r_star)

# 重放值迭代：与 TODO 1 同一条更新式，但把每一轮的 v_k 都留下来
K_REPLAY = 200
v_track = [np.zeros(NUM_STATES)]
v = np.zeros(NUM_STATES)
for _ in range(K_REPLAY):
    q = R + GAMMA * np.einsum("sat,t->sa", T, v)
    v = q.max(axis=1)
    v_track.append(v.copy())
v_track = np.array(v_track)                      # (K_REPLAY+1, 16)：第 k 行是 v_k

d0 = float(np.abs(v_track[1] - v_track[0]).max())          # ‖v_0 − T*v_0‖∞
ks = np.arange(len(v_track))
bound = GAMMA ** ks / (1 - GAMMA) * d0                     # 预测：γᵏ/(1−γ)·‖v₀−v₁‖
measured = np.abs(v_track - v_star_exact).max(axis=1)      # 实测：‖v_k − v*‖∞

print(f"‖v_0 − T*v_0‖∞ = {d0:.4f}")
print("\n k     预测上界         实测误差      实测/上界")
for k in (0, 1, 2, 5, 10, 20, 50, 100, 150, 200):
    print(f"{k:>3}  {bound[k]:>12.4e}  {measured[k]:>12.4e}  {measured[k] / bound[k]:>9.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.semilogy(ks, bound, lw=2, label=r"bound  $\gamma^k/(1-\gamma)\cdot\|v_0-v_1\|_\infty$")
ax.semilogy(ks, measured, lw=2, ls="--", label=r"measured  $\|v_k-v^*\|_\infty$")
ax.set_xlabel("iteration $k$")
ax.set_ylabel("error (log scale)")
ax.set_title(f"Predicted bound vs measured error ($\\gamma$ = {GAMMA})")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
# ✅ 自检 3：上界逐轮包住实测误差
violations = measured > bound * (1 + 1e-9) + 1e-12
assert not violations.any(), f"误差界被击穿：k = {np.where(violations)[0][:5]}"

tight = float((measured / bound).max())
print(f"实测/上界的最大比值 = {tight:.6f} —— 界不但逐轮包住实测误差，而且极紧")
print("✅ 自检通过：γᵏ/(1−γ)·‖v₀−T*v₀‖∞ ≥ ‖v_k − v*‖∞ 对 k = 0..200 全部成立")

EPS = 1e-8
guarantee = GAMMA / (1 - GAMMA) * EPS
print(f"\n实践换算：以「相邻两轮差 < ε = {EPS:.0e}」停机 ⇒ 真实误差 ≤ γ/(1−γ)·ε = {guarantee:.1e}")

## 9 · 回顾：你刚刚审计了什么

- **实现**：值迭代（贪心自举）与策略迭代（PE 线性求解 + PI 贪心改进）——第四讲两张算法伪代码的可执行版；
- **对账**：两法 v\* 差 < 1e-6，定理 4.1「殊途同归」的承诺兑现；
- **收敛形状**：值迭代是 log 轴上的直线（每轮 ×γ），策略迭代是几个大步到底（有限步终止）；
- **γ 扫描**：迭代轮数随 γ 单调上升（本世界 28 → 176 → 1834），策略却纹丝不动——γ 定尺度，名次定策略；
- **误差界**：γᵏ/(1−γ)·‖v₀−v₁‖ 逐轮包住实测误差且极紧——ε 换算公式可以放心用。

## 🏔 挑战：策略迭代步数少，但每步更贵——总账怎么算？

值迭代每轮便宜（一次 $O(n^2|A|)$ 的张量收缩）但要几百轮；策略迭代只要 5 轮外层，但每轮 PE 是一次 $O(n^3)$ 的线性求解。第四讲 §4.3 给出的答案是折中——**截断策略迭代**：把评估步截断到 j 步，j = 1 退化为值迭代，j 充分大就是策略迭代，总计算量的最优点常在中间。

你的任务：用一段代码或一段分析，验证（或反驳）这个权衡。三个方向任选：

- **A · 计时**：`time.perf_counter` 分别测两法总耗时，重复 ≥ 20 次取中位数（浏览器计时噪声大；更稳的口径是数 FLOP：solve 的 $O(n^3)$ vs 张量收缩的 $O(n^2|A|)$）；
- **B · 截断策略迭代**：把 PE 截断成 j 步迭代评估，扫 j ∈ {1, 2, 4, 8, ∞}，画「总更新次数 vs j」，找总计算量最小的折中点；
- **C · 规模**：把世界扩成 6×6 / 8×8（改 `SIZE` 重造 T/R），观察两法轮数与耗时随 n 的增长——$O(n^3)$ 与 $O(n^2)$ 的赛跑谁先撑不住？

In [ ]:
# 🏔 挑战区（无标准答案，不影响跑绿）——把你的实验写在下面，替换这一行：
print("🏔 挑战区：写下你的实验（本格保持可运行即可，不参与自检）")

## 🎓 下一本

到这里三处 TODO 与全部 ✅ 自检格跑绿——第四讲的两条定理已被你逐轮审计完毕：压缩映射给了「每轮 ×γ」的速率与唯一不动点，误差界给了可信的停机换算。

下一本 **NB3 · 蒙特卡洛与随机近似**（L5/L6）：世界从「动力学已知」翻到「动力学未知」——不能再 einsum 算期望了，只能采样，方差第一次成为一等公民。